# 03. Temporal Snapshot Construction

## Objective

This notebook creates the modeling timeline. One snapshot represents one observed customer at one monthly `ReferenceDate`. Customer status uses only purchases and full cancellations known at that date. The prediction target is defined only for customers who are active at the reference date: it indicates whether the customer reaches 60 consecutive days without a valid purchase during the following 30 days. Reactivation is outside the current scope.

In [ ]:
import pandas as pd

customer_transactions = pd.read_parquet(
    "../data/interim/customer_transactions.parquet"
)

print("Rows:", len(customer_transactions))
print("Customers:", customer_transactions["Customer ID"].nunique())
display(customer_transactions["InvoiceDate"].agg(["min", "max"]))
customer_transactions.head()

## 1. Conservative full-reversal matching

Cancellation invoices do not directly identify the original purchase invoice. Each cancellation product is therefore linked to the closest preceding product within 30 days only when customer, `StockCode`, absolute quantity, and value are identical. A purchase invoice is treated as fully reversed only when at least 99% of both its quantity and value are matched. Partial or uncertain cancellations remain separate customer interactions and do not erase the purchase.

In [ ]:
positive_products = customer_transactions.loc[
    ~customer_transactions["Invoice"].str.startswith("C")
].copy()
positive_products["PurchaseLineValue"] = (
    positive_products["Quantity"] * positive_products["Price"]
)
positive_products = (
    positive_products
    .groupby(["Customer ID", "Invoice", "StockCode"], as_index=False)
    .agg(
        PurchaseDate=("InvoiceDate", "max"),
        PurchasedQuantity=("Quantity", "sum"),
        PurchaseLineValue=("PurchaseLineValue", "sum"),
    )
    .loc[lambda rows: rows["PurchasedQuantity"] > 0]
    .rename(columns={"Invoice": "PurchaseInvoice"})
)
positive_products["MatchQuantity"] = positive_products["PurchasedQuantity"]
positive_products["MatchValue"] = positive_products["PurchaseLineValue"].round(2)

cancellation_products = customer_transactions.loc[
    customer_transactions["Invoice"].str.startswith("C")
].copy()
cancellation_products["CancelledQuantity"] = -cancellation_products["Quantity"]
cancellation_products["CancellationValue"] = (
    -cancellation_products["Quantity"] * cancellation_products["Price"]
)
cancellation_products = (
    cancellation_products
    .groupby(["Customer ID", "Invoice", "StockCode"], as_index=False)
    .agg(
        CancellationDate=("InvoiceDate", "max"),
        CancelledQuantity=("CancelledQuantity", "sum"),
        CancellationValue=("CancellationValue", "sum"),
    )
    .loc[lambda rows: rows["CancelledQuantity"] > 0]
    .rename(columns={"Invoice": "CancellationInvoice"})
)
cancellation_products["MatchQuantity"] = cancellation_products["CancelledQuantity"]
cancellation_products["MatchValue"] = cancellation_products["CancellationValue"].round(2)

product_matches = pd.merge_asof(
    cancellation_products.sort_values("CancellationDate"),
    positive_products.sort_values("PurchaseDate"),
    left_on="CancellationDate",
    right_on="PurchaseDate",
    by=["Customer ID", "StockCode", "MatchQuantity", "MatchValue"],
    direction="backward",
    tolerance=pd.Timedelta(days=30),
)
product_matches = (
    product_matches.dropna(subset=["PurchaseInvoice"])
    .sort_values("CancellationDate")
    .drop_duplicates(["Customer ID", "PurchaseInvoice", "StockCode"])
)

purchase_totals = (
    positive_products.groupby(["Customer ID", "PurchaseInvoice"], as_index=False)
    .agg(
        PurchaseDate=("PurchaseDate", "max"),
        GrossInvoiceQuantity=("PurchasedQuantity", "sum"),
        GrossInvoiceValue=("PurchaseLineValue", "sum"),
    )
)
matched_totals = (
    product_matches.groupby(["Customer ID", "PurchaseInvoice"], as_index=False)
    .agg(
        FullCancellationDate=("CancellationDate", "max"),
        MatchedQuantity=("CancelledQuantity", "sum"),
        MatchedValue=("CancellationValue", "sum"),
    )
)
invoice_reversal_audit = purchase_totals.merge(
    matched_totals, on=["Customer ID", "PurchaseInvoice"], how="left"
)
invoice_reversal_audit[["MatchedQuantity", "MatchedValue"]] = (
    invoice_reversal_audit[["MatchedQuantity", "MatchedValue"]].fillna(0)
)
invoice_reversal_audit["QuantityReversedShare"] = (
    invoice_reversal_audit["MatchedQuantity"]
    / invoice_reversal_audit["GrossInvoiceQuantity"] * 100
)
invoice_reversal_audit["ValueReversedShare"] = (
    invoice_reversal_audit["MatchedValue"]
    / invoice_reversal_audit["GrossInvoiceValue"].where(
        invoice_reversal_audit["GrossInvoiceValue"] != 0, 1
    ) * 100
)
invoice_reversal_audit["PotentialFullReversal"] = (
    invoice_reversal_audit["QuantityReversedShare"].ge(99)
    & (
        invoice_reversal_audit["GrossInvoiceValue"].eq(0)
        | invoice_reversal_audit["ValueReversedShare"].ge(99)
    )
)
full_reversal_invoices = invoice_reversal_audit.loc[
    invoice_reversal_audit["PotentialFullReversal"]
].copy()

print("Potential fully reversed invoices:", len(full_reversal_invoices))
print("Customers concerned:", full_reversal_invoices["Customer ID"].nunique())

## 2. Reusable invoice event tables

The transaction history is reduced to one row per invoice. Non-`C` invoices are purchase candidates. `FullCancellationDate` records when a conservative full reversal became known, so the event is valid before that timestamp and invalid from that timestamp onward. Cancellation invoices are stored separately for later behavioral features.

In [ ]:
purchase_lines = customer_transactions.loc[
    ~customer_transactions["Invoice"].str.startswith("C")
].copy()
purchase_lines["LineValue"] = purchase_lines["Quantity"] * purchase_lines["Price"]
purchase_lines["IsManual"] = purchase_lines["StockCode"].eq("M")
purchase_lines["IsZeroPrice"] = purchase_lines["Price"].eq(0)

purchase_invoice_events = (
    purchase_lines.groupby(["Customer ID", "Invoice"], as_index=False)
    .agg(
        PurchaseDate=("InvoiceDate", "max"),
        Country=("Country", "first"),
        InvoiceValue=("LineValue", "sum"),
        InvoiceQuantity=("Quantity", "sum"),
        ProductLines=("StockCode", "size"),
        UniqueProducts=("StockCode", "nunique"),
        HasManual=("IsManual", "max"),
        HasZeroPrice=("IsZeroPrice", "max"),
    )
    .merge(
        full_reversal_invoices[[
            "Customer ID", "PurchaseInvoice", "FullCancellationDate"
        ]],
        left_on=["Customer ID", "Invoice"],
        right_on=["Customer ID", "PurchaseInvoice"],
        how="left",
    )
    .drop(columns="PurchaseInvoice")
)

cancellation_lines = customer_transactions.loc[
    customer_transactions["Invoice"].str.startswith("C")
].copy()
cancellation_lines["CancelledQuantity"] = -cancellation_lines["Quantity"]
cancellation_lines["CancellationValue"] = (
    -cancellation_lines["Quantity"] * cancellation_lines["Price"]
)
cancellation_invoice_events = (
    cancellation_lines.groupby(["Customer ID", "Invoice"], as_index=False)
    .agg(
        CancellationDate=("InvoiceDate", "max"),
        CancellationValue=("CancellationValue", "sum"),
        CancelledQuantity=("CancelledQuantity", "sum"),
        ProductLines=("StockCode", "size"),
        UniqueProducts=("StockCode", "nunique"),
    )
)

print("Purchase invoices:", len(purchase_invoice_events))
print("Cancellation invoices:", len(cancellation_invoice_events))

## 3. Monthly reference dates

All customers share a reference date on the first day of each month. The first retained date follows at least 120 days of global transaction history, which gives the initial features a meaningful observation window. The final date leaves the complete 30-day prediction horizon inside the dataset. The 120 historical days are a feature-coverage choice, not the churn threshold.

In [ ]:
churn_threshold_days = 60
prediction_horizon_days = 30
minimum_history_days = 120

first_purchase_date = purchase_invoice_events["PurchaseDate"].min()
observation_end = customer_transactions["InvoiceDate"].max()

reference_dates = pd.DataFrame({
    "ReferenceDate": pd.date_range(
        start=first_purchase_date.normalize() + pd.offsets.MonthBegin(1),
        end=observation_end - pd.Timedelta(days=prediction_horizon_days),
        freq="MS",
    )
})
reference_dates["HistoryDaysAvailable"] = (
    reference_dates["ReferenceDate"] - first_purchase_date
).dt.days
reference_dates["FutureDaysAvailable"] = (
    observation_end - reference_dates["ReferenceDate"]
).dt.days
reference_dates = reference_dates.loc[
    reference_dates["HistoryDaysAvailable"] >= minimum_history_days
].reset_index(drop=True)

print("Reference dates:", len(reference_dates))
print("First reference date:", reference_dates["ReferenceDate"].min())
print("Last reference date:", reference_dates["ReferenceDate"].max())
reference_dates

## 4. Customer population at each reference date

A customer enters a snapshot only after at least one purchase is known and still valid at that date. Purchases on the reference date belong to the future period, so historical events must satisfy `PurchaseDate < ReferenceDate`. A future cancellation cannot modify an earlier snapshot.

In [ ]:
customer_reference_calendar = (
    purchase_invoice_events[["Customer ID"]].drop_duplicates()
    .merge(reference_dates[["ReferenceDate"]], how="cross")
)
purchase_history = customer_reference_calendar.merge(
    purchase_invoice_events[[
        "Customer ID", "PurchaseDate", "FullCancellationDate"
    ]],
    on="Customer ID",
)
purchase_history = purchase_history.loc[
    purchase_history["PurchaseDate"].lt(purchase_history["ReferenceDate"])
    & (
        purchase_history["FullCancellationDate"].isna()
        | purchase_history["FullCancellationDate"].gt(
            purchase_history["ReferenceDate"]
        )
    )
]
customer_snapshots = (
    purchase_history.groupby(["Customer ID", "ReferenceDate"], as_index=False)
    .agg(
        FirstObservedPurchaseDate=("PurchaseDate", "min"),
        LastPurchaseDate=("PurchaseDate", "max"),
    )
    .sort_values(["ReferenceDate", "Customer ID"])
    .reset_index(drop=True)
)

print("Snapshot rows:", len(customer_snapshots))
print("Customers:", customer_snapshots["Customer ID"].nunique())
customer_snapshots.head()

## 5. Churn status at the reference date

`ChurnDeadline` is exactly 60 days after the latest valid purchase. A customer is churned when the reference timestamp reaches that deadline. `RecencyDays` is retained as a continuous feature, while status is calculated from timestamps to avoid rounding at the boundary.

In [ ]:
customer_snapshots["RecencyDays"] = (
    customer_snapshots["ReferenceDate"]
    - customer_snapshots["LastPurchaseDate"]
).dt.total_seconds() / 86400
customer_snapshots["ChurnDeadline"] = (
    customer_snapshots["LastPurchaseDate"]
    + pd.Timedelta(days=churn_threshold_days)
)
customer_snapshots["IsChurnedAtReference"] = (
    customer_snapshots["ReferenceDate"]
    >= customer_snapshots["ChurnDeadline"]
)

status_by_reference = (
    customer_snapshots.groupby("ReferenceDate")["IsChurnedAtReference"]
    .agg(Customers="size", Churned="sum")
)
status_by_reference["Active"] = (
    status_by_reference["Customers"] - status_by_reference["Churned"]
)
status_by_reference["ChurnRate"] = (
    status_by_reference["Churned"] / status_by_reference["Customers"] * 100
)
status_by_reference.round(2)

## 6. Churn during the next 30 days

The target is evaluated only for customers active at the reference date. `WillChurnNext30Days = True` when the customer reaches a 60-day inactivity deadline inside the half-open window `[ReferenceDate, OutcomeEndDate)`, where `OutcomeEndDate = ReferenceDate + 30 days`. An event exactly at `OutcomeEndDate` belongs to the next snapshot, not to the current target. A valid purchase before the deadline prevents the event. The calculation also checks full-cancellation timestamps inside the outcome window because a newly known reversal can reveal that the latest remaining valid purchase is already older than 60 days.

Only two kinds of timestamps can create a churn event: a purchase deadline and a full-cancellation date. The helper therefore checks those exact timestamps rather than generating one row per customer per day.

In [ ]:
events_by_customer = {
    customer_id: events.sort_values("PurchaseDate").reset_index(drop=True)
    for customer_id, events in purchase_invoice_events.groupby("Customer ID")
}

def first_churn_event(snapshot):
    start = snapshot["ReferenceDate"]
    end = start + pd.Timedelta(days=prediction_horizon_days)
    events = events_by_customer[snapshot["Customer ID"]]

    checkpoints = pd.concat([
        events["PurchaseDate"] + pd.Timedelta(days=churn_threshold_days),
        events["FullCancellationDate"],
    ]).dropna()
    checkpoints = checkpoints.loc[
        checkpoints.gt(start) & checkpoints.lt(end)
    ].sort_values().unique()

    for checkpoint in pd.to_datetime(checkpoints):
        valid_purchases = events.loc[
            events["PurchaseDate"].lt(checkpoint)
            & (
                events["FullCancellationDate"].isna()
                | events["FullCancellationDate"].gt(checkpoint)
            )
        ]
        if valid_purchases.empty:
            return checkpoint
        latest_valid_purchase = valid_purchases["PurchaseDate"].max()
        if latest_valid_purchase + pd.Timedelta(days=churn_threshold_days) <= checkpoint:
            return checkpoint

    return pd.NaT

labeled_snapshots = customer_snapshots.copy()
labeled_snapshots["OutcomeEndDate"] = (
    labeled_snapshots["ReferenceDate"]
    + pd.Timedelta(days=prediction_horizon_days)
)
active_mask = ~labeled_snapshots["IsChurnedAtReference"]
labeled_snapshots["ChurnEventDate"] = pd.NaT
labeled_snapshots.loc[active_mask, "ChurnEventDate"] = (
    labeled_snapshots.loc[active_mask].apply(first_churn_event, axis=1)
)
labeled_snapshots["WillChurnNext30Days"] = (
    labeled_snapshots["ChurnEventDate"].notna().astype("boolean")
)
labeled_snapshots.loc[~active_mask, "WillChurnNext30Days"] = pd.NA

labeled_snapshots["LabelEndDate"] = pd.NaT
labeled_snapshots.loc[active_mask, "LabelEndDate"] = (
    labeled_snapshots.loc[active_mask, "OutcomeEndDate"]
)
positive_target = labeled_snapshots["WillChurnNext30Days"].eq(True).fillna(False)
labeled_snapshots.loc[positive_target, "LabelEndDate"] = (
    labeled_snapshots.loc[positive_target, "ChurnEventDate"]
)

labeled_snapshots["ObservedTenureDays"] = (
    labeled_snapshots["ReferenceDate"]
    - labeled_snapshots["FirstObservedPurchaseDate"]
).dt.total_seconds() / 86400
labeled_snapshots.head()

### 6.1 Development target by reference date

The modeling denominator is the number of active customers at each reference date. The target rate answers one operational question: among customers who can still be retained today, what proportion will cross the 60-day inactivity boundary during the next 30 days? Only dates before the final test period are displayed.

In [ ]:
target_by_reference = (
    labeled_snapshots.loc[
        active_mask
        & labeled_snapshots["ReferenceDate"].lt("2011-09-01")
    ]
    .groupby("ReferenceDate")["WillChurnNext30Days"]
    .agg(ActiveSnapshots="size", ChurnsNext30Days="sum")
)
target_by_reference["ChurnRateNext30Days"] = (
    target_by_reference["ChurnsNext30Days"]
    / target_by_reference["ActiveSnapshots"] * 100
)
target_by_reference.round(2)

## 7. Structural and temporal validation

Every check below must return zero. `LabelEndDate` is the last timestamp used to know the outcome: the churn-event timestamp for positive targets and the end of the 30-day window for negative targets. This column will drive purging during temporal validation.

In [ ]:
core_columns = [
    "Customer ID",
    "FirstObservedPurchaseDate",
    "ReferenceDate",
    "LastPurchaseDate",
    "RecencyDays",
    "ObservedTenureDays",
    "ChurnDeadline",
    "IsChurnedAtReference",
]
active_mask = ~labeled_snapshots["IsChurnedAtReference"]
churned_mask = labeled_snapshots["IsChurnedAtReference"]
positive_target = labeled_snapshots["WillChurnNext30Days"].eq(True).fillna(False)
negative_target = labeled_snapshots["WillChurnNext30Days"].eq(False).fillna(False)

validation_checks = pd.Series({
    "Duplicate customer-reference rows": labeled_snapshots.duplicated(
        ["Customer ID", "ReferenceDate"]
    ).sum(),
    "Missing core values": labeled_snapshots[core_columns].isna().any(axis=1).sum(),
    "Last purchase not before reference": (
        labeled_snapshots["LastPurchaseDate"]
        >= labeled_snapshots["ReferenceDate"]
    ).sum(),
    "Negative recency": labeled_snapshots["RecencyDays"].lt(0).sum(),
    "Incorrect churn deadline": (
        labeled_snapshots["ChurnDeadline"]
        != labeled_snapshots["LastPurchaseDate"]
        + pd.Timedelta(days=churn_threshold_days)
    ).sum(),
    "Incorrect reference status": (
        labeled_snapshots["IsChurnedAtReference"]
        != (labeled_snapshots["ReferenceDate"] >= labeled_snapshots["ChurnDeadline"])
    ).sum(),
    "Missing target for active snapshots": labeled_snapshots.loc[
        active_mask, "WillChurnNext30Days"
    ].isna().sum(),
    "Target assigned to churned snapshots": labeled_snapshots.loc[
        churned_mask, "WillChurnNext30Days"
    ].notna().sum(),
    "Positive target without event date": labeled_snapshots.loc[
        positive_target, "ChurnEventDate"
    ].isna().sum(),
    "Negative target with event date": labeled_snapshots.loc[
        negative_target, "ChurnEventDate"
    ].notna().sum(),
    "Churn event outside outcome window": (
        positive_target
        & (
            labeled_snapshots["ChurnEventDate"]
            >= labeled_snapshots["OutcomeEndDate"]
        )
    ).sum(),
    "Invalid label end for positive target": (
        positive_target
        & (labeled_snapshots["LabelEndDate"] != labeled_snapshots["ChurnEventDate"])
    ).sum(),
    "Invalid label end for negative target": (
        negative_target
        & (labeled_snapshots["LabelEndDate"] != labeled_snapshots["OutcomeEndDate"])
    ).sum(),
    "Incomplete outcome windows": (
        labeled_snapshots["OutcomeEndDate"] > observation_end
    ).sum(),
}, name="Errors").to_frame()
validation_checks["Status"] = validation_checks["Errors"].eq(0).map({
    True: "PASS", False: "CHECK"
})
validation_checks

## 8. Final test and model-selection folds

September through November 2011 form the untouched out-of-time test. Earlier snapshots form the development period. Inside development, the two approved selection approaches are retained: one temporal holdout and two expanding walk-forward folds. A training row is retained when its `LabelEndDate` is no later than `ValidationStart`. Equality is safe because the target window excludes `OutcomeEndDate`, while validation features use only events strictly before their reference date. There is no fixed 60-day or 120-day embargo because each row is checked from its exact label-observation cutoff, which is at most 30 days after the reference date.

In [ ]:
labeled_snapshots["FinalSplit"] = "development"
labeled_snapshots.loc[
    labeled_snapshots["ReferenceDate"].between("2011-09-01", "2011-11-01"),
    "FinalSplit",
] = "test"

fold_definitions = [
    {
        "Approach": "A",
        "Method": "single_temporal_holdout",
        "Fold": "holdout_1",
        "TrainStart": "2010-04-01",
        "ValidationStart": "2011-01-01",
        "ValidationEnd": "2011-02-01",
    },
    {
        "Approach": "B",
        "Method": "purged_walk_forward",
        "Fold": "fold_1",
        "TrainStart": "2010-04-01",
        "ValidationStart": "2010-10-01",
        "ValidationEnd": "2010-11-01",
    },
    {
        "Approach": "B",
        "Method": "purged_walk_forward",
        "Fold": "fold_2",
        "TrainStart": "2010-04-01",
        "ValidationStart": "2011-01-01",
        "ValidationEnd": "2011-02-01",
    },
]
model_selection_folds = pd.DataFrame(fold_definitions)
date_columns = ["TrainStart", "ValidationStart", "ValidationEnd"]
model_selection_folds[date_columns] = model_selection_folds[date_columns].apply(
    pd.to_datetime
)

active_labeled = labeled_snapshots.loc[
    ~labeled_snapshots["IsChurnedAtReference"]
].copy()
fold_counts = []
for fold in model_selection_folds.itertuples():
    candidate_train = active_labeled["ReferenceDate"].between(
        fold.TrainStart, fold.ValidationStart, inclusive="left"
    )
    train = candidate_train & active_labeled["LabelEndDate"].le(
        fold.ValidationStart
    )
    validation = active_labeled["ReferenceDate"].between(
        fold.ValidationStart, fold.ValidationEnd
    )
    fold_counts.append({
        "CandidateTrainRows": candidate_train.sum(),
        "PurgedTrainRows": (candidate_train & ~train).sum(),
        "TrainRows": train.sum(),
        "ValidationRows": validation.sum(),
        "NoLabelOverlap": active_labeled.loc[train, "LabelEndDate"].max()
        <= fold.ValidationStart,
    })
model_selection_folds = pd.concat([
    model_selection_folds, pd.DataFrame(fold_counts)
], axis=1)
model_selection_folds

In [ ]:
split_summary = (
    active_labeled.groupby("FinalSplit")
    .agg(
        ReferenceDates=("ReferenceDate", "nunique"),
        Snapshots=("Customer ID", "size"),
        Customers=("Customer ID", "nunique"),
    )
)
split_summary

## 9. Export validated modeling inputs

The snapshot table, fold definitions, and reusable invoice events are exported for EDA and point-in-time feature engineering. Predictive features have not yet been calculated.

In [ ]:
snapshots_path = "../data/interim/labeled_customer_snapshots.parquet"
folds_path = "../data/interim/model_selection_folds.csv"
purchase_events_path = "../data/interim/purchase_invoice_events.parquet"
cancellation_events_path = "../data/interim/cancellation_invoice_events.parquet"

labeled_snapshots.to_parquet(snapshots_path, index=False)
model_selection_folds.to_csv(folds_path, index=False)
purchase_invoice_events.to_parquet(purchase_events_path, index=False)
cancellation_invoice_events.to_parquet(cancellation_events_path, index=False)

print("Snapshots saved:", snapshots_path, labeled_snapshots.shape)
print("Folds saved:", folds_path, model_selection_folds.shape)
print("Purchase events saved:", purchase_events_path, purchase_invoice_events.shape)
print("Cancellation events saved:", cancellation_events_path, cancellation_invoice_events.shape)